In [11]:
import math
import random

In [ ]:
class Mathlib:
    def randn(*shape):
        pass

    def clip(val, minVal=0, maxVal=1):
        return(min(maxVal, max(minVal, val)))

    def mean(vals):
        return(sum(vals)/len(vals))
    
    def argmax(vals):
        maxVal = vals[0]
        maxIndex = 0
        for i, v in enumerate(vals):
            if v > maxVal:
                maxVal = v
                maxIndex = i
        return(maxIndex)
    
    def argmin(vals):
        minVal = vals[0]
        minIndex = 0
        for i, v in enumerate(vals):
            if v < minVal:
                minVal = v
                minIndex = i
        return(minIndex)

    def dot2Vectors(vector1, vector2):
        out = 0
        for v1, v2 in zip(vector1, vector2):
            out += v1*v2
        return(out)

    def normalize(values):
        out = []
        total = sum(values)
        for v in values:
            out.append(v/total)
        return(out)

In [26]:
class Activation:
    def RecLE(val):
        return(max(0, val))    #< by clipping 0, you de-linearize your data
    def RecLE_dx(val):
        """ partial derivative of max(x,0) with respect to x """
        return(1 if val > 0 else 0)

    def none(val):
        return(val)
    def none_dx(val):
        """ derivative of x with respect to x """
        return(1)

    def softmax(val):
        return(math.e**val)    #< useful to ensure no negative values
    def softmax_dx(val):
        """ derivative of e^x with respect to x """
        return(math.e**val)    

    def protectedSoftmax(vals):
        """ softmax, but between 0 and 1 """
        out = []
        maxVal = max(vals)
        for v in vals:
            out.append(Activation.softmax(v-maxVal))
        return(out)
    def protectedSoftmax_dx(vals):
        """ derivative of e^x with respect to x as protected softmax is just regular softmax with extra steps to ensure values are normalized """
        return(Activation.softmax_dx(vals))


In [ ]:
class Loss:
    def crossEntropyLoss_single(val):
        return(-math.log(Mathlib.clip(val, 1e-7, 1-1e-7)))

    def crossEntropyLoss(vals, targets):
        """calculate the cross entropy loss of a softmax output

        Args:
            vals (list): results
            targets (list): target results
        """
        out = 0
        for v, t in zip(vals, targets):
            out += t*Loss.crossEntropyLoss_single(v)   #< loss times the weight of the loss, more important = bigger loss
        return(out/len(vals))


In [23]:
class Accuracy_hard:
    def __init__(self):
        self._length = 0
        self.correct = 0
        self.accuracy = 0.0
    def calc(self, result, target):
        if type(target) == list:
            target = Mathlib.argmax(target)
        if Mathlib.argmax(result) == target:
            self.correct += 1
        self._length += 1
        self.accuracy = self.correct/self._length
        return(self.accuracy)

class Accuracy_soft:
    def __init__(self):
        self._length = 0
        self.correct = 0.0
        self.accuracy = 0.0
    def calc(self, result, target):
        dot = Mathlib.dot2Vectors(result, target)
        norm_r = math.sqrt(sum(r*r for r in result))
        norm_t = math.sqrt(sum(t*t for t in target))
        soft = dot / (norm_r * norm_t)      #< normalize between 0 and 1

        self._length += 1
        self.correct += soft
        self.accuracy = self.correct/self._length
        return(self.accuracy)
        

In [15]:
class Neuron:
    """ A single neuron, stores its inputs, weights, bias, and activation function """
    def __init__(self, inputs, weights, bias, activationFunc=Activation.none):
        self.inputs = inputs      #< all outputs from previous layer
        self.weights = weights    #< one weight for every input
        self.bias = bias          #< one bias per neuron
        self.activationFunc = activationFunc

    def calc(self):
        """
        Compute the neuron's output.
        Multiply each input by its corresponding weight, sum the
        results, add the bias and then apply the activation function.
        
        multiply the inputs [i1, i2.. in] with weights [w1, w1... wn] to get i1*w1+i2*w2... +in*wn, then add the bias
        """
        
        return(self.activationFunc(Mathlib.dot2Vectors(self.inputs, self.weights)+self.bias))


In [ ]:
class Layer:
    """A collection of neurons forming a single layer in the network"""
    def __init__(self, inputs, weights, biases, normalize=True, layerActivation=Activation.none, *args, **kwargs):
        self.out = None
        self.normalize = normalize
        self.layerActivation = layerActivation  #< layer activation function
        self.neurons = self._createNeurons(inputs, weights, biases, *args, **kwargs)  # create neuron objects

    def _createNeurons(self, inputs, weights, biases, *args, **kwargs):
        neurons = []
        for w, b in zip(weights, biases):
            neurons.append(Neuron(inputs, w, b, *args, **kwargs))
        return(neurons)
    
    def calc(self):
        """Calculate outputs for all neurons in the layer, apply layer-level
        activation and optional normalization."""
        self.out = [neuron.calc() for neuron in self.neurons]
        self.out = self.layerActivation(self.out)      #< Apply layer activation function (e.g., softmax, protectedSoftmax)
        if self.normalize:
            self.out = Mathlib.normalize(self.out)     #< convert output from 0 to 1
        return(self.out)

    def calcLoss(self, target):
        """Convenience wrapper for computing loss using either a hot target, or a distribution of targets"""
        if type(target) == list:
            return(Loss.crossEntropyLoss(self.out, target))
            
        return(Loss.crossEntropyLoss_single(self.out[target]))


In [ ]:
class Batch:
    """ Process multiple samples at once in a batch."""
    def __init__(self, inputsBatch, *args, **kwargs):
        self.batch = self._createBatch(inputsBatch, *args, **kwargs)  # create a Layer for each sample

    def _createBatch(self, inputsBatch, *args, **kwargs):
        """Helper method to create a Layer object for each sample in the batch."""
        batch = []
        for inputs in inputsBatch:
            batch.append(Layer(inputs, *args, **kwargs))  # instantiate Layer and add to batch
        return(batch)
    
    def calc(self):
        """Run all layers in the batch and return their outputs."""
        return([layer.calc() for layer in self.batch])
    
    def calcLoss(self, correctIndexes):
        """Compute mean loss across the batch given a list of target indexes."""
        return(Mathlib.mean([self.batch[i].calcLoss(correctIndexes[i]) for i in range(len(self.batch))]))  # average of per-sample losses


In [ ]:
class Train:
    """ TBD """
    def __init__(self, weights, biases, learningRate=0.01):
        self.weights = weights
        self.biases = biases
        self.learningRate = learningRate

    def step(self, inputs, correctIndex, activation=Activation.protectedSoftmax):
        """Perform a single gradient descent update for one example.

        Args:
            inputs: list of input features for the sample
            correctIndex: index of the true class in the output vector
            activation: activation used at layer level (default softmax variant)
        """
        layer = Layer(inputs, self.weights, self.biases, layerActivation=activation)
        out = layer.calc()
        for i in range(len(self.weights)) :
            # target probability is 1 for correct class, 0 otherwise (eg: 0 for cat, 0 for dog, 1 for horse)
            target = 0
            if i == correctIndex:
                target = 1
  
            # protectedSoftmax produces an output of e^x/(sum(e^x))
            # the error function is -log(result)
            # the error is -sum(targetX*log(e^x/(sum(e^x)))
            # for target with 0s and 1s [0, 0, 1, 0], the error simplifies to -log(e^x/(sum(e^x))
            # to know if the error has improved, we need to take the derivative of the error
            # the derivative of -log(e^x/(sum(e^x)) simplifies to result-target
            error = out[i] - target
            for j in range(len(self.weights[i])):
                # learning rate determines the size of the change
                # the error determines how far away we are, therefore the size of the update
                # inputs determines the importance of that input
                # an smaller input that has less effect, will be changed less
                self.weights[i][j] -= self.learningRate * error * inputs[j]
            # learning rate determines the size of the change
            # the error determines how far away we are, therefore the size of the update
            # since we update both the weights and biases, we might overcompensate, which is ok because there are multiple iterations
            self.biases[i] -= self.learningRate * error

    def train(self, inputsBatch, correctIndexes, epochs=1, activation=Activation.protectedSoftmax):
        """Train for a number of epochs """
        for _ in range(epochs):
            for inputs, correct in zip(inputsBatch, correctIndexes):
                self.step(inputs, correct, activation)

In [ ]:
import numpy as np
np.random.seed(0)

def createSetData():
    inputs = [[1.0, 2.0, 3.0, 2.5],
              [2.0, 5.0, -1.0, 2.0],
              [-1.5, 2.7, 3.3, -0.8]]
    weights = np.array([[0.2, 0.8, -0.5, 1.0],
               [0.5, -0.91, 0.26, -0.5],
               [-0.26, -0.27, 0.17, 0.87]])

    biases = [2.0, 3.0, 0.5]
    print("expected output:")
    print(np.dot(inputs, weights.T) + biases)
    print("-------------------------\n\n\n")
    return(inputs, weights, biases)

def createData(n_inputs, n_neurons, batchSize=1):
    inputs = np.random.randn(batchSize, n_neurons)
    weights = 0.01*np.random.randn(n_inputs, n_neurons)
    biases = np.zeros(n_neurons)
    return(inputs, weights, biases)

inputs, weights, biases = createSetData()

batch = Batch(inputs, weights, biases, layerActivation=Activation.protectedSoftmax)
# batch = Batch(inputs, weights, biases, activationFunc=Activation.softmax, layerActivation=Activation.none)
out = batch.calc()
print(np.array(out))

# print(Loss.crossEntropyLoss_single(0.7))
# print(Loss.crossEntropyLoss_single(0.1))
# print(Loss.crossEntropyLoss_single(0.08))

print(batch.calcLoss([0,0,0]))

expected output:
[[ 4.8    1.21   2.385]
 [ 8.9   -1.81   0.2  ]
 [ 1.41   1.051  0.026]]
-------------------------



[[8.95282664e-01 2.47083068e-02 8.00090293e-02]
 [9.99811129e-01 2.23163963e-05 1.66554348e-04]
 [5.13097164e-01 3.58333899e-01 1.28568936e-01]]
0.35667494393873245
2.3025850929940455
0.10536051565782628
0.25936490708526483
